In [0]:
from pyspark.sql.functions import *

In [0]:
df_prov_hospA = spark.read.option('header', True)\
                .option('inderschema', True)\
                .parquet('abfss://bronze@adlshosptial.dfs.core.windows.net/EMR/HospitalA/providers')

df_prov_hospB =  spark.read.option('header', True)\
                .option('inferschema', True)\
                .parquet('abfss://bronze@adlshosptial.dfs.core.windows.net/EMR/HospitalB/providers')

df_prov = df_prov_hospA.unionByName(df_prov_hospB)

df_prov = df_prov.withColumn('FullName',\
                 concat(col('FirstName'), lit(' '), col('LastName')))\
                    .withColumn('datasource',regexp_replace(col('datasource'),'-','_'))

df_prov= df_prov.withColumn('Src_Dept_Id', col('DeptID'))\
                    .withColumn('Dept_ID', concat(col('DeptID'),lit('_'),col('datasource')))\
                    .drop('DeptID')
df_prov.createOrReplaceTempView('Providers')

display(df_prov)

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS silver;

CREATE TABLE IF NOT EXISTS silver.providers(
ProviderID VARCHAR(200),
FirstName VARCHAR(200),
LastName VARCHAR(200),
Specialization VARCHAR(200),
NPI VARCHAR(200),
datasource VARCHAR(200),
FullName VARCHAR(200),
Src_Dept_Id VARCHAR(200),
Dept_ID VARCHAR(200),
is_quarantined BOOLEAN
)
USING DELTA;




In [0]:
%sql
TRUNCATE TABLE silver.providers;

INSERT INTO silver.providers
    SELECT * , 
    CASE
        WHEN FULLNAME is NULL THEN True
        ELSE False
    END
    FROM Providers;

In [0]:
%sql
select * from silver.providers